In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'usecase':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC_PATH = PROJECT_ROOT / 'src'
if str(SRC_PATH) not in sys.path:
    sys.path.insert(0, str(SRC_PATH))


# Use Case 1: Corporate Future Borrowing Hedge

## Business Scenario

A corporate treasurer expects to refinance or synthetically lock in fixed-rate borrowing in **2 years** for a **5-year** horizon.
The concern is that fixed rates may rise before the borrowing date. One practical hedge is to buy a **2Y x 5Y payer swaption**,
which gives the right to enter a pay-fixed swap if future swap rates move higher.

This notebook prices that hedge, studies the main Greeks, and checks how much protection it provides under rate-shock scenarios.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from swaption_pricing.black76 import price_swaption
from swaption_pricing.data_loader import load_curve_points_csv
from swaption_pricing.hedging import compare_model_hedging
from swaption_pricing.market_validation import load_json_metadata
from swaption_pricing.risk import compare_all_model_risks
from swaption_pricing.sabr import SabrParams, price_swaption_with_sabr
from swaption_pricing.swap import forward_swap_rate, swap_annuity
from swaption_pricing.types import SwaptionSpec


## Market Inputs

This use case uses the public U.S. Treasury yield-curve proxy already fetched into the project.
That means the curve is **market-connected**, but still a **proxy** rather than a full dealer-grade SOFR swap curve.

For the volatility side, we use:

- Black benchmark volatility: `20%`
- SABR parameters: `alpha=0.02`, `beta=0.5`, `rho=-0.25`, `nu=0.40`

These vol inputs are still project assumptions rather than vendor swaption quotes.

In [ ]:
curve_path = PROJECT_ROOT / 'data/raw/market/ust_yield_curve_proxy/curve_points.csv'
metadata_path = PROJECT_ROOT / 'data/raw/market/ust_yield_curve_proxy/ust_yield_curve_snapshot.json'

curve = load_curve_points_csv(curve_path)
curve_metadata = load_json_metadata(metadata_path)
curve_metadata

## Trade Specification

Representative hedge trade:

- Notional: `USD 50,000,000`
- Expiry: `2Y`
- Underlying swap tenor: `5Y`
- Option type: `payer`
- Strike: chosen at current ATM forward swap rate

In [ ]:
notional = 50_000_000.0
expiry = 2.0
tenor = 5.0
pay_frequency = 1
option_type = 'payer'

atm_strike = forward_swap_rate(curve, expiry, tenor, pay_frequency)
spec = SwaptionSpec(
    notional=notional,
    expiry=expiry,
    tenor=tenor,
    strike=atm_strike,
    pay_frequency=pay_frequency,
    option_type=option_type,
)

black_vol = 0.20
normal_vol = 0.01
shift = 0.03
sabr_params = SabrParams(alpha=0.0200, beta=0.50, rho=-0.25, nu=0.40)

trade_inputs = pd.DataFrame([
    {
        'curve_snapshot_date': curve_metadata['yield_curve_date'],
        'notional': notional,
        'expiry': expiry,
        'tenor': tenor,
        'option_type': option_type,
        'atm_strike': atm_strike,
    }
])
trade_inputs

## Pricing Results

We price the same hedge under five frameworks:

- Black
- SABR-adjusted Black
- shifted Black
- shifted SABR
- Bachelier

This is useful even in a corporate hedging case because the premium the treasurer pays depends on the model and volatility convention.

In [ ]:
comparison = compare_all_model_risks(
    curve=curve,
    spec=spec,
    black_vol=black_vol,
    sabr_params=sabr_params,
    shift=shift,
    normal_vol=normal_vol,
)

pricing_df = pd.DataFrame([
    {
        'model': comparison.black.model,
        'price': comparison.black.price,
        'volatility_input': comparison.black.volatility,
    },
    {
        'model': comparison.sabr.model,
        'price': comparison.sabr.price,
        'volatility_input': comparison.sabr.volatility,
    },
    {
        'model': comparison.shifted_black.model,
        'price': comparison.shifted_black.price,
        'volatility_input': comparison.shifted_black.volatility,
    },
    {
        'model': comparison.shifted_sabr.model,
        'price': comparison.shifted_sabr.price,
        'volatility_input': comparison.shifted_sabr.volatility,
    },
    {
        'model': comparison.bachelier.model,
        'price': comparison.bachelier.price,
        'volatility_input': comparison.bachelier.volatility,
    },
])
pricing_df

In [ ]:
forward = forward_swap_rate(curve, expiry, tenor, pay_frequency)
annuity = swap_annuity(curve, expiry, tenor, pay_frequency)
sabr_price, sabr_implied_vol = price_swaption_with_sabr(curve, spec, sabr_params)

trade_summary = pd.DataFrame([
    {
        'forward_swap_rate': forward,
        'annuity': annuity,
        'black_price': price_swaption(curve, spec, black_vol),
        'sabr_price': sabr_price,
        'sabr_implied_vol': sabr_implied_vol,
    }
])
trade_summary

## Risk Results

For a corporate hedge, the premium matters, but so do the risk sensitivities:

- `PV01`: how sensitive the hedge value is to rate moves
- `vega`: how sensitive the premium is to volatility assumptions
- `theta`: how the option decays over time

In [ ]:
risk_df = pd.DataFrame([
    {
        'model': comparison.black.model,
        'pv01': comparison.black.risk.pv01,
        'vega': comparison.black.risk.vega,
        'theta': comparison.black.risk.theta,
    },
    {
        'model': comparison.sabr.model,
        'pv01': comparison.sabr.risk.pv01,
        'vega': comparison.sabr.risk.vega,
        'theta': comparison.sabr.risk.theta,
    },
    {
        'model': comparison.shifted_black.model,
        'pv01': comparison.shifted_black.risk.pv01,
        'vega': comparison.shifted_black.risk.vega,
        'theta': comparison.shifted_black.risk.theta,
    },
    {
        'model': comparison.shifted_sabr.model,
        'pv01': comparison.shifted_sabr.risk.pv01,
        'vega': comparison.shifted_sabr.risk.vega,
        'theta': comparison.shifted_sabr.risk.theta,
    },
    {
        'model': comparison.bachelier.model,
        'pv01': comparison.bachelier.risk.pv01,
        'vega': comparison.bachelier.risk.vega,
        'theta': comparison.bachelier.risk.theta,
    },
])
risk_df

In [ ]:
ax = pricing_df.plot(x='model', y='price', kind='bar', figsize=(9, 4), title='Premium by Model')
ax.set_ylabel('Price')
plt.xticks(rotation=20)
plt.show()

ax = risk_df.plot(x='model', y=['pv01', 'vega', 'theta'], kind='bar', figsize=(10, 4), title='Risk Comparison by Model')
ax.set_ylabel('Sensitivity')
plt.xticks(rotation=20)
plt.show()

## Scenario Analysis and Simple Hedge

A treasurer often wants to know two things:

1. How much the hedge gains if rates rise
2. Whether a simple swap-based PV01 hedge changes the residual PnL profile

Below we compare unhedged and hedged PnL under a `+25bp` parallel shift.

In [ ]:
rate_shift = 0.0025
hedging_comparison = compare_model_hedging(
    curve=curve,
    spec=spec,
    black_vol=black_vol,
    sabr_params=sabr_params,
    shift=shift,
    normal_vol=normal_vol,
    rate_shift=rate_shift,
)

hedge_df = pd.DataFrame([
    {
        'model': hedging_comparison.black.model,
        'hedge_ratio': hedging_comparison.black.hedge_ratio,
        'unhedged_pnl': hedging_comparison.black.unhedged_pnl,
        'hedged_pnl': hedging_comparison.black.hedged_pnl,
    },
    {
        'model': hedging_comparison.sabr.model,
        'hedge_ratio': hedging_comparison.sabr.hedge_ratio,
        'unhedged_pnl': hedging_comparison.sabr.unhedged_pnl,
        'hedged_pnl': hedging_comparison.sabr.hedged_pnl,
    },
    {
        'model': hedging_comparison.shifted_black.model,
        'hedge_ratio': hedging_comparison.shifted_black.hedge_ratio,
        'unhedged_pnl': hedging_comparison.shifted_black.unhedged_pnl,
        'hedged_pnl': hedging_comparison.shifted_black.hedged_pnl,
    },
    {
        'model': hedging_comparison.shifted_sabr.model,
        'hedge_ratio': hedging_comparison.shifted_sabr.hedge_ratio,
        'unhedged_pnl': hedging_comparison.shifted_sabr.unhedged_pnl,
        'hedged_pnl': hedging_comparison.shifted_sabr.hedged_pnl,
    },
    {
        'model': hedging_comparison.bachelier.model,
        'hedge_ratio': hedging_comparison.bachelier.hedge_ratio,
        'unhedged_pnl': hedging_comparison.bachelier.unhedged_pnl,
        'hedged_pnl': hedging_comparison.bachelier.hedged_pnl,
    },
])
hedge_df

In [ ]:
ax = hedge_df.plot(x='model', y=['unhedged_pnl', 'hedged_pnl'], kind='bar', figsize=(10, 4), title='Corporate Hedge PnL under +25bp Rate Shock')
ax.set_ylabel('PnL')
plt.xticks(rotation=20)
plt.show()

## Interpretation

- This use case shows how a payer swaption can act as insurance against higher future fixed borrowing costs.
- The option premium depends on model choice, especially once smile-aware or low-rate frameworks are considered.
- The hedge gains value when rates move higher, which is precisely the exposure the corporate treasurer wants to offset.
- The simple swap-based PV01 hedge reduces first-order rate exposure, but it does not remove all model dependence.

## Practical Limitations

- The curve is a public-data Treasury proxy, not a dealer-grade SOFR swap curve.
- The volatility inputs are still project assumptions rather than a real market swaption vol slice.
- The rate-shock study is a simplified parallel-shift scenario rather than a full scenario engine.